# Тема 7 — Тюнинг vLLM: двигаем саму кривую

## Профиль — это гипотеза с доказательством

**Что случилось в Topic 06.** Вы нашли колено насыщения своего сервиса на T4:
волна 64, ~460 ток/с, дальше — зона, где лишние клиенты покупают очередь, а не токены.
Вы выбрали рабочую точку и защитили её письменно. Отлично.

**Проблема:** рабочая точка — это максимум того, что даёт конфигурация по умолчанию.
А если другая конфигурация даст на 20% больше токенов при том же SLO? Или вдвое
лучший хвост? Вы не знаете — потому что не двигали кривую.

**Что меняется сейчас.** У кривой насыщения есть ручки. `--max-num-seqs` ограничивает
батч decode. `--gpu-memory-utilization` делит VRAM между весами и KV-пулом.
`--max-model-len` объявляет потолок контекста. Chunked prefill режет чужой prefill
на части. Каждая ручка управляет механизмом, который вы уже объяснили в Topic 06 —
batching, KV-бюджет, интерференция prefill.

**Ваша задача на лабу.** (1) Предсказать эффект каждой ручки ДО измерений — таблица
«ручка → механизм → ожидание → риск». (2) Объявить пространство поиска: 5 конфигураций,
не больше, каждая с одно-предложённой причиной. (3) Прогнать все 5 через те же волны
8/32/64/128 с тем же журналом. (4) Построить Парето-фронт «скорость против хвоста»,
выбрать профиль под SLO и заморозить с прогнозом. (5) Проверить на held-out текстах.
(6) Stretch: краевой тест и план отката.

**Прецедент курса — Round 03.** Владелец поднял бар производительности на 2.3%:
FLASH_ATTN + `--max-num-seqs 64`. Попутно был отвергнут Triton-кандидат —
quality gate упал ниже пола. Отвергнутый кандидат остался отвергнутым:
повторный запуск ради «удачного» числа — это не поиск, а подгонка. Этот стандарт — ваш.

**Правила игры.** (1) Неудачная конфигурация — строка в таблице, не вычёркиваемый
эксперимент. (2) План замораживается ДО sweep — как прогноз колена в Topic 06.
(3) После заморозки профиль не меняется; held-out судит выбор, а не Beauty-contest.
(4) Профиль без плана отката — это ставка без выхода.

Профили: **mock** (CPU, минуты, объявленная синтетика) и **gpu** (T4, instructor).
Числа mock — не измерения vLLM. Конверт: 2 контактных часа + 3–5 самостоятельных.

## ⚡ Как переключить на GPU (обязательно для measured-результатов)

**По умолчанию этот ноутбук работает на бесплатном CPU (mock-профиль).**
Все числа mock — объявленная синтетика. Для measured-данных нужен GPU-прогон.

**Тяжёлые ожидания — в самом начале:** сразу после setup стоят `t07-gpu-bootstrap`
(ставит vLLM и поднимает сервер) и `t07-gpu-health` (ждёт готовности). Запустите
верхние клетки первыми — установка (~4 мин) и загрузка модели (~4 мин) пойдут,
пока вы разбираете теорию ручек. Финал ноутбука — реальный эксперимент:
default против вашего замороженного профиля на волнах судьи 32/64.

**Пошагово:**

1. **Runtime → Change runtime type → T4 GPU** (High-RAM выключить)
2. **Runtime → Restart session** (сбросить всё состояние)
3. Скопируйте и выполните ЭТУ клетку первой (перед t07-setup):
```python
import os
os.environ['LSM_T07_PROFILE'] = 'gpu'
print('GPU профиль активирован — далее Run all')
```
4. **Ctrl+F9 (Run all)**: bootstrap + health ~8 мин (идут фоном, пока вы на
   теории), мок-поиск ~2 мин, финальный эксперимент на реальном сервере
   ~6–12 мин (один перезапуск сервера с вашим профилем + волны 32/64)
5. После `t07-export` — **скачайте bundle и .ipynb** (File → Download) ДО отключения!

**Без шага 3 вы получите mock-числа, даже если GPU подключён.**
Если установка vLLM не удалась с первого раза (сетевой таймаут) — ноутбук
автоматически retry'ит до 3 раз. Если все 3 не удались — перезапустите
runtime и попробуйте ещё раз.

In [ ]:
import json, os, time, threading, hashlib, urllib.request
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from typing import TypedDict, Sequence
from tqdm.auto import tqdm  # правило курса: видимый прогресс в каждой клетке дольше ~5 с

PROFILE = os.environ.get('LSM_T07_PROFILE', os.environ.get('LSM_T06_PROFILE', 'mock'))  # 'mock' | 'gpu' (owner-run only)
ATTEMPT = 'attempt-001'
RESULTS = os.path.join('results', PROFILE, ATTEMPT)
os.makedirs(RESULTS, exist_ok=True)
print(f'profile={PROFILE}  results dir={RESULTS}')

In [ ]:
# GPU PROFILE ONLY. Pinned vLLM install + owned child service in one bootstrap.
# Стоит ПЕРВОЙ парой клеток: установка и загрузка модели идут, пока вы разбираете
# теорию ручек — к финальному эксперименту сервер уже горячий.
# Оценка времени: pip ~2–5 мин на попытку (тихо, флаг -q; до 3 попыток при сетевых
# сбоях), итог печатается ОДНОЙ строкой. Готовность сервера — следующая клетка.
if PROFILE != 'gpu':
    print('skipped: gpu-only cell (profile is mock)')
else:
    import subprocess  # noqa: E402
    out = None
    _attempts = 0
    for _attempt in range(3):
        _attempts = _attempt + 1
        out = subprocess.run(['pip', 'install', '-q', 'vllm==0.30.0', '--timeout', '120'],
                             capture_output=True, text=True)
        if out.returncode == 0:
            break
        time.sleep(5)
    assert out.returncode == 0, ('vLLM install failed: ' + (out.stderr or '')[-300:])
    SERVER_CMD = ['python', '-m', 'vllm.entrypoints.openai.api_server',
                  '--model', 'Qwen/Qwen2.5-3B-Instruct',
                  '--revision', 'aa8e72537993ba99e69dfaafa59ed015b17504d1',
                  '--served-model-name', 'submission', '--dtype', 'float16',
                  '--max-model-len', '2048', '--port', '8017']
    server = subprocess.Popen(SERVER_CMD,
                              stdout=open(os.path.join(RESULTS, 'server.log'), 'w'),
                              stderr=subprocess.STDOUT)
    print(f'pip: vllm==0.30.0 установлен с попытки {_attempts}; '
          f'server pid {server.pid} — следующая клетка ждёт /health')

In [ ]:
# GPU PROFILE ONLY. Bounded readiness wait; never wave into a dead server.
# Оценка времени: обычно 60–180 с (загрузка весов 3B + CUDA-графы); потолок 900 с.
if PROFILE != 'gpu':
    print('skipped: gpu-only cell (profile is mock)')
else:
    import urllib.error  # noqa: E402
    waited = 0
    healthy = False
    deadline = time.monotonic() + 900
    with tqdm(total=900, desc='ждём /health', unit=' с', unit_scale=False) as bar:
        while not healthy:
            try:
                with urllib.request.urlopen('http://127.0.0.1:8017/health', timeout=5) as r:
                    healthy = r.status == 200
            except (urllib.error.URLError, OSError):
                pass
            assert time.monotonic() < deadline, 'service did not become healthy in 900 s'
            if not healthy:
                time.sleep(5); waited += 5; bar.update(5)
    print(f'service healthy; waited about {waited} s')

In [ ]:
class ModelConfig(TypedDict):
    """Serving-relevant model facts, declared once and shared by every calculation."""
    n_layers: int; n_kv_heads: int; head_dim: int; dtype_bytes: int

MODEL_FACTS: ModelConfig = dict(n_layers=36, n_kv_heads=2, head_dim=128, dtype_bytes=2)

class RequestRecord(TypedDict):
    """One client-observed request. Timestamps are client-side monotonic seconds."""
    request_id: str; wave: int; client_id: int; app_id: str; case_id: str
    input_tokens: int; max_tokens: int
    arrival_ts: float; first_byte_ts: float | None; terminal_ts: float
    output_tokens: int; http_status: int | None; outcome: str

class Percentiles(TypedDict):
    """Nearest-rank TTFT percentiles for one wave, completed requests only."""
    wave: int; count: int; p50_ms: float; p95_ms: float

class AuditSummary(TypedDict):
    """Damage classification of one wave's journal; nothing is repaired."""
    records: int; unique_ids: int; missing_terminal: int; duplicate_ids: int
    foreign_waves: int; impossible_time: int; complete: int

class Decision(TypedDict):
    """The chosen operating wave with its evidence and named binding constraint."""
    wave: int; throughput_tok_per_s: float; ttft_p95_ms: float; error_rate: float
    satisfies_slo: bool; reason: str

class InterferenceReport(TypedDict):
    """A-app degradation when B shares the server, plus B's completed count."""
    a_throughput_ratio: float; a_ttft_p95_ratio: float
    a_completed_alone: int; a_completed_shared: int; b_completed_shared: int

def save_jsonl(path: str, rows: Sequence[dict]) -> str:
    """Write a JSONL journal and return its sha256 hex digest.

    Parameters
    ----------
    path : str
        Destination file path, overwritten.
    rows : sequence of dict
        Records to serialize, one JSON object per line.

    Returns
    -------
    str
        sha256 of the written bytes, for provenance receipts.
    """
    with open(path, 'w', encoding='utf-8') as fh:
        for row in rows:
            fh.write(json.dumps(row, ensure_ascii=False) + chr(10))
    return hashlib.sha256(Path_bytes(path)).hexdigest()

def Path_bytes(path: str) -> bytes:
    """Read a file as raw bytes.

    Parameters
    ----------
    path : str
        File to read.

    Returns
    -------
    bytes
        Exact file content, for hashing.
    """
    with open(path, 'rb') as fh:
        return fh.read()

def load_jsonl(path: str) -> list[dict]:
    """Load a JSONL file into a list of dicts.

    Parameters
    ----------
    path : str
        Journal file, one JSON object per line.

    Returns
    -------
    list of dict
        Parsed records, in file order.
    """
    with open(path, encoding='utf-8') as fh:
        return [json.loads(line) for line in fh if line.strip()]

In [ ]:
# Данные, которыми занимается сервис: живые тексты замороженного корпуса.
# корпус — публичный файл курса (pinned-ревизия); качаем один раз и кэшируем рядом.
import pandas as pd
from IPython.display import display
PINNED_CORPUS_URL = (
    'https://raw.githubusercontent.com/levshaazz/'
    'llm-serving-mastery-submission-example/'
    '8525bd29959562db69800bf453555155f2ade668/classroom/topic06-purpose-v4/corpus.jsonl')
if not os.path.exists('corpus.jsonl'):
    urllib.request.urlretrieve(PINNED_CORPUS_URL, 'corpus.jsonl')
corpus_rows = [json.loads(l) for l in open('corpus.jsonl', encoding='utf-8') if l.strip()]
# берём 3 тикета A и 1 документ B — ровно то, чем будет нагружаться сервис
sample = ([r for r in corpus_rows if r['app_id'] == 'A'][:3]
          + [r for r in corpus_rows if r['app_id'] == 'B'][:1])
_tbl = pd.DataFrame([{'app': r['app_id'], 'case_id': r['case_id'], 'gold': r['gold'],
                      'tokens': r['input_tokens'],
                      'текст (начало)': r['user_text'][:80] + '…'} for r in sample])
display(_tbl.style.set_caption(
    f'всего {len(corpus_rows)} текстов замороженного корпуса;'
    ' A-тикет ≈ 100 токенов, B-документ ≈ 500; gold — правильная метка, не обещание модели'))

In [ ]:
# Та же нагрузка глазами длин: где живут A и B.
import matplotlib
try:
    get_ipython()  # интерактивный фронтенд (Colab/Jupyter): инлайн-бэкенд, графики видны
except NameError:
    matplotlib.use('Agg')  # headless-запуск: только файлы
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 3.2))
# две полупрозрачные гистограммы на общих осях: где живут длины A и B
for app, color in (('A', '#2563eb'), ('B', '#d97706')):
    lens = [r['input_tokens'] for r in corpus_rows if r['app_id'] == app]
    ax.hist(lens, bins=30, alpha=0.55, label=f'приложение {app} (n={len(lens)})', color=color)
ax.set_xlabel('длина входа, токены')
ax.set_ylabel('число текстов')
ax.set_title('Замороженный корпус: A-тикет против B-документа')
ax.legend(); fig.tight_layout(); plt.show()

## Блок 0 — Ручки vLLM: от механизма к параметру

В Topic 06 вы объясняли механизмы через картинку: batching экономит чтение весов,
KV-пул ограничивает число резидентов, prefill крадёт шаг у decode. Теперь у каждого
механизма есть именованная ручка — параметр запуска vLLM.

**Что вы должны увидеть в таблице ниже:**

| Ручка | Механизм из Topic 06 | Что двигает на кривой |
|---|---|---|
| `--max-num-seqs` | Потолок батча decode | Амортизация vs хвост |
| `--gpu-memory-utilization` | Доля VRAM под KV-пул | Потолок резидентов |
| `--max-model-len` | Планирование worst-case | Зарезервированная память |
| chunked prefill | Нарезка чужого prefill | TPOT резидентов |

**Ваша задача** — для каждой ручки написать:
- `expected_effect`: что произойдёт с кривой насыщения (одним-двумя предложениями)
- `risk`: чем вы жертвуете (одним предложением)

**Подсказки:**
- `--max-num-seqs 32`: меньше параллельных decode → каждый тик быстрее (меньше
  конкуренции за compute), но меньше запросов на батч → амортизация весов хуже
- `--gpu-memory-utilization 0.85`: меньше VRAM под KV → раньше наступает preemption,
  но больше места для активаций → меньше риск OOM
- `--max-model-len 1024`: scheduler планирует под короткие промпты → меньше
  зарезервированной памяти, но длинный промпт будет отклонён
- chunked prefill = True: prefill нового запроса делится на части → decode не
  останавливается на целый шаг → TPOT резидентов плавнее

Таблица **замораживается в файл ДО любых измерений** — как прогноз колена в Topic 06.
Когда измерение не совпадёт с предсказанием — это самые интересные данные, потому
что они указывают на механизм, который вы не учли.

In [ ]:
class MockTuningServer:
    """Расширенный мок с ручками vLLM: seqs, mem, chunked. Объявленная синтетика.

    max_num_seqs ограничивает эффективный параллелизм (эмулирует потолок batch).
    gpu_mem_util масштабирует ёмкость (меньше = раньше preemption).
    chunked разбивает prefill на шаги (decode не голодает один шаг).
    """
    def __init__(self, max_num_seqs: int = 256, gpu_mem_util: float = 0.9,
                 chunked: bool = False) -> None:
        """Bind the tuning knobs for this mock server instance.

        Parameters
        ----------
        max_num_seqs : int, default 256
            Ceiling on effective decode batch size.
        gpu_mem_util : float, default 0.9
            Fraction of VRAM available for the KV pool.
        chunked : bool, default False
            Whether large prefills are sliced across steps.

        Returns
        -------
        None
            Stateful helper; see request().
        """
        self.max_num_seqs = max_num_seqs
        self.gpu_mem_util = gpu_mem_util
        self.chunked = chunked
        self._lock = threading.Lock()
        self._in_flight = 0

    def request(self, case: dict, max_tokens: int) -> tuple[str, int]:
        """Serve one synthetic request under the configured knobs.

        Parameters
        ----------
        case : dict
            Load case with input_tokens.
        max_tokens : int
            Requested output tokens.

        Returns
        -------
        tuple of (str, int)
            Output text and output token count.
        """
        input_tokens = case['input_tokens']
        with self._lock:
            n = self._in_flight = self._in_flight + 1
        try:
            eff = min(n, self.max_num_seqs)
            queue_ms = 0.08 * eff * eff / max(0.5, self.gpu_mem_util)
            prefill_ms = 0.3 * input_tokens / (2.0 if self.chunked else 1.0)
            ttft = 0.15 + queue_ms / 1000 + prefill_ms / 1000
            per_tok = 0.002 * (1.0 + 0.06 * (eff - 1)) + 0.001 * 8.0 / max(1, eff)
            time.sleep(ttft)
            time.sleep(per_tok * max_tokens)
            return 'SYNTHETIC-MOCK-OUTPUT', max_tokens
        finally:
            with self._lock:
                self._in_flight -= 1

def run_wave(runner: object, app_id: str, cases: Sequence[dict], wave: int, per_client: int,
             timeout_s: float) -> list[RequestRecord]:
    """Run one closed-loop wave of `wave` clients, `per_client` requests each.

    Parameters
    ----------
    runner : object
        Service with .request(case, max_tokens); mock here, HTTP on gpu.
    app_id : str
        Application label written into every record.
    cases : sequence of dict
        Load cases with case_id and input_tokens (mock) or rendered (gpu).
    wave : int
        Concurrency: number of simultaneous closed-loop clients.
    per_client : int
        Sequential requests each client sends.
    timeout_s : float
        Per-request wall budget before the client gives up.

    Returns
    -------
    list of RequestRecord
        One record per terminated attempt, outcomes classified.
    """
    records: list[RequestRecord] = []
    lock = threading.Lock()

    def client(cid: int) -> None:
        """Send this client sequential requests and journal each attempt.

        Parameters
        ----------
        cid : int
            Client index inside the wave.

        Returns
        -------
        None
            Results are appended to the shared records list.
        """
        for seq in range(per_client):
            case = cases[(cid * per_client + seq) % len(cases)]
            rid = f'w{wave}-c{cid}-r{seq}'
            t0 = time.monotonic()
            outcome, status, out_tok, fb = 'connection_error', None, 0, None
            try:
                text, out_tok = runner.request(case, 32)
                fb = time.monotonic()
                outcome, status = ('completed', 200) if out_tok > 0 else ('truncated', 200)
                if time.monotonic() - t0 > timeout_s:
                    outcome = 'timeout'
            except TimeoutError:
                outcome = 'timeout'
            except Exception as exc:  # noqa: BLE001 - transport failures are data
                outcome = 'server_error' if '5xx' in str(exc) else 'connection_error'
            rec = RequestRecord(request_id=rid, wave=wave, client_id=cid, app_id=app_id,
                                case_id=case['case_id'], input_tokens=case['input_tokens'],
                                max_tokens=32, arrival_ts=t0, first_byte_ts=fb,
                                terminal_ts=time.monotonic(), output_tokens=out_tok,
                                http_status=status, outcome=outcome)
            if time.monotonic() - t0 > timeout_s and outcome == 'completed':
                rec['outcome'] = 'timeout'
            with lock:
                records.append(rec)

    with ThreadPoolExecutor(max_workers=wave) as pool:
        list(pool.map(client, range(wave)))
    return records

def run_shared_wave(server: object, b_cap_k: int, a_cases: Sequence[dict], b_cases: Sequence[dict],
                    n_a: int, n_b: int, per_client: int, timeout_s: float) -> list[RequestRecord]:
    """One concurrent wave: n_a A-clients plus n_b B-clients capped at b_cap_k.

    Parameters
    ----------
    server : object
        Runner shared by both applications; overlap IS the interference.
    b_cap_k : int
        In-flight cap for B, one of 1/2/4/8/16.
    a_cases : sequence of dict
        Load cases for application A.
    b_cases : sequence of dict
        Load cases for application B.
    n_a : int
        A client count.
    n_b : int
        B client count.
    per_client : int
        Sequential requests per client.
    timeout_s : float
        Per-request wall budget.

    Returns
    -------
    list of RequestRecord
        Merged A+B journal; app_id and the B- request-id prefix distinguish them.
    """
    capped = InFlightCapRunner(server, b_cap_k)
    with ThreadPoolExecutor(max_workers=n_a + n_b) as pool:
        fut_a = pool.submit(run_wave, server, 'A', a_cases, n_a, per_client, timeout_s)
        fut_b = pool.submit(run_wave, capped, 'B', b_cases, n_b, per_client, timeout_s)
        a_recs, b_recs = fut_a.result(), fut_b.result()
    for r in b_recs:
        r['request_id'] = 'B-' + r['request_id']
    return a_recs + b_recs

In [ ]:
def _wave_stats(records: Sequence[RequestRecord], w: int) -> dict:
    """Compute per-wave summary statistics for the table.

    Parameters
    ----------
    records : sequence of RequestRecord
        Full journal.
    w : int
        Wave label.

    Returns
    -------
    dict
        n, completed, timeouts, wall_s, throughput, p50_ms, p95_ms.
    """
    rs = [r for r in records if r['wave'] == w]
    done = [r for r in rs if r['outcome'] == 'completed']
    wall = max(r['terminal_ts'] for r in rs) - min(r['arrival_ts'] for r in rs)
    tt = sorted((r['first_byte_ts'] - r['arrival_ts']) * 1000
                for r in done if r['first_byte_ts'])
    return dict(wave=w, n=len(rs), completed=len(done),
                timeouts=sum(1 for r in rs if r['outcome'] == 'timeout'),
                wall_s=round(wall, 2),
                throughput=round(sum(r['output_tokens'] for r in done) / wall, 1),
                p50_ms=round(tt[len(tt) // 2]) if tt else None,
                p95_ms=round(tt[-(-95 * len(tt) // 100) - 1]) if tt else None)

def summary_rows(records: Sequence[RequestRecord], waves: Sequence[int]) -> 'pd.DataFrame':
    """Build the per-wave summary table as a DataFrame indexed by wave.

    Parameters
    ----------
    records : sequence of RequestRecord
        Full journal.
    waves : sequence of int
        Wave labels to include.

    Returns
    -------
    pandas.DataFrame
        One row per wave with outcomes and timing columns.
    """
    import pandas as pd
    return pd.DataFrame([_wave_stats(records, w) for w in waves]).set_index('wave')

def plot_curve(xs: Sequence[int], ys: Sequence[float], ylabel: str, path: str,
               title: str = '', slo_ms: float | None = None,
               knee: int | None = None,
               p50s: Sequence[float] | None = None) -> None:
    """One annotated curve: value labels, optional SLO line and knee marker.

    Parameters
    ----------
    xs : sequence of int
        Concurrency values.
    ys : sequence of float
        Metric values, same length as xs.
    ylabel : str
        Y-axis label with units.
    path : str
        File name inside RESULTS.
    title : str, default ''
        Optional figure title.
    slo_ms : float or None, default None
        SLO reference line in milliseconds.
    knee : int or None, default None
        Knee wave to annotate with an arrow.
    p50s : sequence of float or None, default None
        Optional second series (p50) drawn as a lighter line with a legend.

    Returns
    -------
    None
        Writes the PNG into RESULTS and shows it.
    """
    fig, ax = plt.subplots(figsize=(7, 4.4))
    ax.plot(xs, ys, marker='o', color='#2563eb', label='p95')
    if p50s is not None:
        ax.plot(xs, p50s, marker='o', color='#93c5fd', label='p50')
        ax.legend(loc='center left')
    for i, (x, y) in enumerate(zip(xs, ys)):
        # p95-подпись сверху; при двух сериях p50-подпись снизу и только если линии разошлись
        ax.annotate(f'{y:.4g}', (x, y), textcoords='offset points', xytext=(0, 9),
                    ha='center', fontsize=9)
        if p50s is not None and ys[i] and p50s[i] / ys[i] < 0.82:
            ax.annotate(f'{p50s[i]:.4g}', (x, p50s[i]), textcoords='offset points',
                        xytext=(0, -14), ha='center', fontsize=8, color='#64748b')
    if slo_ms is not None:
        ax.axhline(slo_ms, color='#dc2626', linestyle='--', lw=1.4)
        ax.annotate(f'SLO {slo_ms:g}', xy=(1.0, slo_ms), xycoords=('axes fraction', 'data'),
                    ha='right', va='bottom', color='#dc2626', fontsize=9)
    if knee is not None and knee in list(xs):
        yk = ys[list(xs).index(knee)]
        ax.annotate('колено', (knee, yk), textcoords='offset points', xytext=(14, -20),
                    color='#15803d', fontsize=10,
                    arrowprops=dict(arrowstyle='->', color='#15803d'))
    ax.margins(x=0.10)
    ax.set_xlabel('конкурентность (закрытый цикл)')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(RESULTS, path), dpi=120, bbox_inches='tight')
    plt.show()

def outcome_shares(records: Sequence[RequestRecord], waves: Sequence[int], path: str) -> None:
    """Stacked per-wave outcome shares with percent labels; all records counted.

    Parameters
    ----------
    records : sequence of RequestRecord
        Full journal.
    waves : sequence of int
        Wave labels to plot.
    path : str
        File name inside RESULTS.

    Returns
    -------
    None
        Writes the PNG into RESULTS and shows it.
    """
    kinds = ['completed', 'timeout', 'server_error', 'connection_error', 'truncated']
    fig, ax = plt.subplots(figsize=(6.5, 4))
    bottoms = [0.0] * len(waves)
    for kind in kinds:
        shares = []
        for w in waves:
            rs = [r for r in records if r['wave'] == w]
            shares.append(sum(1 for r in rs if r['outcome'] == kind) / len(rs) if rs else 0.0)
        ax.bar([str(w) for w in waves], shares, bottom=bottoms, label=kind)
        for i, (b, s) in enumerate(zip(bottoms, shares)):
            if s >= 0.06:
                ax.annotate(f'{s * 100:.0f}%', (i, b + s / 2), ha='center',
                            va='center', fontsize=8, color='white')
        bottoms = [b + s for b, s in zip(bottoms, shares)]
    ax.set_xlabel('конкурентность'); ax.set_ylabel('доля завершённых попыток')
    ax.set_title('Исходы по волнам (в знаменателе — каждая попытка)')
    ax.legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc='upper left')
    fig.tight_layout()
    fig.savefig(os.path.join(RESULTS, path), dpi=120, bbox_inches='tight')
    plt.show()

def plot_waterfall(records: Sequence[RequestRecord], wave: int, path: str,
                   max_rows: int = 40) -> None:
    """Request waterfall: grey wait, blue service, red timeout, one bar per request.

    Parameters
    ----------
    records : sequence of RequestRecord
        Full journal.
    wave : int
        Wave label to draw.
    path : str
        File name inside RESULTS.
    max_rows : int, default 40
        Cap on drawn requests, earliest arrivals first.

    Returns
    -------
    None
        Writes the PNG into RESULTS and shows it.
    """
    wave_rs = sorted((r for r in records if r['wave'] == wave),
                     key=lambda r: r['arrival_ts'])
    # равномерная выборка по времени прихода: видны и обслуженные, и таймауты
    step = max(1, len(wave_rs) // max_rows)
    rs = wave_rs[::step][:max_rows]
    t0 = wave_rs[0]['arrival_ts']
    timeout_budget = max((r['terminal_ts'] - r['arrival_ts']) for r in wave_rs
                         if r['outcome'] == 'timeout') if any(
        r['outcome'] == 'timeout' for r in wave_rs) else None
    fig, ax = plt.subplots(figsize=(9.5, max(3.6, 0.26 * len(rs))))
    for i, r in enumerate(rs):
        a = r['arrival_ts'] - t0
        if r['outcome'] == 'timeout' or r['first_byte_ts'] is None:
            ax.barh(i, r['terminal_ts'] - t0 - a, left=a, height=0.7, color='#dc2626')
        else:
            fb = r['first_byte_ts'] - t0
            ax.barh(i, fb - a, left=a, height=0.7, color='#cbd5e1')
            ax.barh(i, r['terminal_ts'] - t0 - fb, left=fb, height=0.7, color='#1d4ed8')
    if timeout_budget:
        ax.axvline(timeout_budget, color='#dc2626', linestyle=':', lw=1.6)
        ax.annotate('бюджет запроса: дальше клиент сдаётся', xy=(timeout_budget, len(rs) - 1),
                    xytext=(6, 0), textcoords='offset points', va='center',
                    color='#dc2626', fontsize=9)
    ax.set_ylim(-1.5, len(rs) + 0.5)
    ax.set_yticks([])
    ax.set_xlabel('секунды от начала волны', labelpad=8)
    n_to = sum(1 for r in wave_rs if r['outcome'] == 'timeout')
    ax.set_title(f'Один запрос = одна строка. Волна {wave}: всего {len(wave_rs)}, '
                 f'таймаутов {n_to}; показаны {len(rs)} равномерно по времени', pad=12)
    ax.text(0.99, 0.02, 'читать: светлый старт строки = ожидание в очереди; '
            'тёмно-синее = обслуживание (decode); красное до пунктира = не дожил',
            transform=ax.transAxes, ha='right', va='bottom', fontsize=8.5,
            bbox=dict(boxstyle='round,pad=0.4', fc='#fffbeb', ec='#f59e0b', alpha=0.9))
    import matplotlib.patches as mpatches
    fig.legend(handles=[mpatches.Patch(color='#cbd5e1', label='ожидание (queue)'),
                        mpatches.Patch(color='#1d4ed8', label='обслуживание (decode)'),
                        mpatches.Patch(color='#dc2626', label='таймаут')],
               fontsize=8, ncol=3, loc='lower center', bbox_to_anchor=(0.5, -0.04))
    fig.tight_layout(rect=[0, 0.05, 1, 1])
    fig.savefig(os.path.join(RESULTS, path), dpi=120, bbox_inches='tight')
    plt.show()

In [ ]:
# GPU PROFILE ONLY. Frozen public corpus (pinned example revision), D/V splits.
if PROFILE != 'gpu':
    print('skipped: gpu-only cell (profile is mock)')
else:
    def _cases(app: str, split: str) -> list[dict]:
        """Select load cases for one application and split.

        Parameters
        ----------
        app : str
            Application label, 'A' or 'B'.
        split : str
            Corpus split, 'D' (development) or 'V' (held-out).

        Returns
        -------
        list of dict
            Cases with case_id, input_tokens and rendered prompt.
        """
        return [dict(case_id=r['case_id'], input_tokens=r['input_tokens'],
                     rendered=r['rendered_text'])
                for r in corpus_rows if r['split'] == split and r['app_id'] == app
                and r['length_bin'] == 'short']
    gpu_cases_A, gpu_cases_B = _cases('A', 'D'), _cases('B', 'D')
    heldout_A, heldout_B = _cases('A', 'V'), _cases('B', 'V')
    print(f'dev A={len(gpu_cases_A)} B={len(gpu_cases_B)}  '
          f'heldout A={len(heldout_A)} B={len(heldout_B)}')

In [ ]:
class HttpRunner:
    """Synchronous completions client for the local pinned service (case dict in)."""
    def __init__(self, port: int = 8017) -> None:
        """Point the client at one local service port.

        Parameters
        ----------
        port : int, default 8017
            Port of the owned vLLM service.

        Returns
        -------
        None
            Connectionless helper; see request().
        """
        self.url = f'http://127.0.0.1:{port}/v1/completions'

    def request(self, case: dict, max_tokens: int) -> tuple[str, int]:
        """Complete one rendered prompt.

        Parameters
        ----------
        case : dict
            Load case with the rendered prompt under 'rendered'.
        max_tokens : int
            Output token cap.

        Returns
        -------
        tuple of (str, int)
            Output text and completion token count.
        """
        body = json.dumps(dict(model='submission', prompt=case['rendered'],
                               max_tokens=max_tokens, temperature=0.0)).encode()
        req = urllib.request.Request(self.url, data=body,
                                     headers={'Content-Type': 'application/json'})
        with urllib.request.urlopen(req, timeout=120) as resp:
            data = json.loads(resp.read())
        return data['choices'][0]['text'], data['usage']['completion_tokens']

In [ ]:
def knob_effect_table() -> list[dict]:
    """Четыре ручки vLLM: механизм и ожидаемый эффект на кривой. Замораживается до измерений.

    Returns
    -------
    list of dict
        По строке на ручку: knob, mechanism, expected_effect, risk.
    """
    # Уровень A: ручки и механизмы вписаны — допишите expected_effect и risk.
    return [
        dict(knob='--max-num-seqs', mechanism='потолок batch decode', expected_effect='TODO', risk='TODO'),
        dict(knob='--gpu-memory-utilization', mechanism='доля VRAM под KV-пул', expected_effect='TODO', risk='TODO'),
        dict(knob='--max-model-len', mechanism='объявленный потолок контекста', expected_effect='TODO', risk='TODO'),
        dict(knob='chunked prefill', mechanism='нарезка чужого prefill', expected_effect='TODO', risk='TODO'),
    ]

_rows = knob_effect_table()
with open(os.path.join(RESULTS, 'knob-table.json'), 'w') as fh:
    json.dump(_rows, fh, ensure_ascii=False, indent=1)
display(pd.DataFrame(_rows).style.set_caption(
    f'таблица ручек заморожена до измерений ({len(_rows)} ручек, knob-table.json)'))

## План поиска: предобъявленный и ограниченный

**Почему нельзя попробовать всё.** 4 ручки × по 3 значения × 4 волны × 3 повтора
= сотни GPU-часов. Бюджет реального проекта — 5 конфигураций, не 500. И этого
достаточно, если правильно выбрать.

**Метод: one-factor-at-a-time (OFAT) + подтверждающая пара.**

1. **Default** (1 конфигурация) — базовая линия, настройки vLLM по умолчанию
2. **3 OFAT** — каждая ручка по очереди от default, остальное не трогаем
3. **1 pair** — два лучших изменения вместе (проверка взаимодействия)

Итого 5. Ни больше, ни меньше.

**Каждая строка — гипотеза с механизмом.** «seqs-32: жертвую амортизацией ради
хвоста» — это предсказание с названным механизмом и направлением. «Попробуем и
посмотрим» — не гипотеза, а бросок монеты с лишними шагами.

**Дисциплина:**
- План замораживается ДО sweep (файл с timestamp и sha256)
- Неудачная конфигурация — строка в таблице, не вычёркиваемый footnote
- Если в середине поиска захотелось попробовать другое — версионируйте план
- Изменение после seeing results — это tuning, не validation

**Прецедент Round 03:** Triton-кандидат выглядел хорошо по скорости, но упал по
quality gate. Он остался отвергнутым. Ваша таблица должна сохранять такие
отвергнутые строки с объяснением, почему они не прошли.

In [ ]:
def declare_search_space() -> list[dict]:
    """Объявить пространство поиска с бюджетом. Замораживается до измерений.

    Returns
    -------
    list of dict
        По строке на конфигурацию: name, max_num_seqs, gpu_mem_util, chunked, reason.
        Не более 6 конфигураций — бюджет ограничен.
    """
    # Уровень B: скелет готов — заполните поля для каждой конфигурации.
    # Шаг 1: включите default (max_num_seqs=256, mem=0.9, chunked=False).
    # Шаг 2: добавьте one-factor-at-a-time: seqs=32, mem=0.85, chunked=True, pair.
    # Шаг 3: для каждой — одно предложение reason.
    raise NotImplementedError('TODO: заполните конфигурации по скелету')

_plan = declare_search_space()
with open(os.path.join(RESULTS, 'search-plan.json'), 'w') as fh:
    json.dump(_plan, fh, ensure_ascii=False, indent=1)
display(pd.DataFrame(_plan).style.set_caption(
    f'план поиска заморожен: {len(_plan)} конфигураций (search-plan.json)'))

In [ ]:
# Константы волн и нагрузка для мок-профиля (перенесены из удалённой t07-run-waves).
WAVES = [8, 32, 64, 128]
MOCK_CASES = [dict(case_id=f'synth-{i}', input_tokens=n)
              for i, n in enumerate([100, 110, 120] * 40)]

# Исполнение поиска: мок-профиль (и на CPU, и на GPU — реальный сервер в поиске
# не участвует). Оценка времени: ~1–2 мин. measured-числа даёт только
# финальный эксперимент в конце ноутбука.
# Неудачные конфигурации — строки таблицы, а не вычёркнутые сноски.
sweep_rows = []
for cfg in tqdm(_plan, desc='sweep', unit='cfg'):
    mock = MockTuningServer(max_num_seqs=cfg['max_num_seqs'],
                            gpu_mem_util=cfg['gpu_mem_util'],
                            chunked=cfg['chunked'])
    recs = []
    for w in tqdm(WAVES, desc=f"  {cfg['name']}", unit=' волна', leave=False):
        recs.extend(run_wave(mock, 'A', MOCK_CASES, w, per_client=3, timeout_s=12.0))
    stats = {w: _wave_stats(recs, w) for w in WAVES}
    sweep_rows.append(dict(
        name=cfg['name'], seqs=cfg['max_num_seqs'], mem=cfg['gpu_mem_util'],
        chunked=cfg['chunked'],
        thr_64=stats[64]['throughput'],
        p95_64=stats[64]['p95_ms'] if stats[64]['p95_ms'] is not None else 99999,
        err_64=stats[64]['timeouts'] / max(1, stats[64]['n']),
    ))
display(pd.DataFrame(sweep_rows))

## Парето-фронт: скорость против хвоста

После sweep каждая конфигурация — точка в двумерном пространстве:
throughput (больше = лучше) × p95 TTFT (меньше = лучше).

**Парето-доминирование:** конфигурация A доминирует B, если A не медленнее И
не грубее, с хотя бы одним строгим превосходством. Фронт = недоминируемые точки.

**Что вы должны увидеть на графике:**
- Синие ромбы = конфигурации на фронте (недоминируемые)
- Красные точки = доминируемые (есть вариант лучше по обоим критериям)
- Зелёная зона слева от SLO-линии = eligible (p95 ≤ 800 мс)
- На фронте нет «лучшего» — есть trade-off, который SLO разрешает

**Как читать фронт:**
1. Найдите точку с максимальным throughput среди eligible
2. Это ваш кандидат для заморозки
3. Проверьте: есть ли на фронте точка чуть медленнее, но значительно мягче?
   Если да — это запас прочности, а не потеря

**Вопрос себе:** если SLO изменится с 800 на 600 мс, какие конфигурации
останутся eligible? Как изменится выбор? Что это говорит о хрупкости решения?

In [ ]:
def pareto_front(rows: Sequence[dict]) -> list[dict]:
    """Отобрать Парето-оптимальные конфигурации: скорость max, хвост min.

    Parameters
    ----------
    rows : sequence of dict
        Строки sweep-таблицы с thr_64 и p95_64.

    Returns
    -------
    list of dict
        Только те строки, где ни одна другая не быстрее И не короче одновременно.
    """
    raise NotImplementedError('TODO: доминирование = быть лучше по обоим критериям')

_front = pareto_front(sweep_rows)
display(pd.DataFrame(_front))

In [ ]:
# Паретовский фронт одной картинкой: скорость против хвоста.
fig, ax = plt.subplots(figsize=(7, 4.5))
for row in sweep_rows:
    color = '#dc2626' if row not in _front else '#2563eb'
    marker = 'o' if row not in _front else 'D'
    ax.scatter(row['p95_64'], row['thr_64'], c=color, marker=marker, s=90, zorder=3)
    ax.annotate(row['name'], (row['p95_64'], row['thr_64']),
                textcoords='offset points', xytext=(8, 4), fontsize=9)
ax.set_xlabel('p95 TTFT на волне 64, мс (меньше — лучше →)')
ax.set_ylabel('throughput на волне 64, ток/с')
ax.set_title('Паретовский фронт: скорость против хвоста (синие ромбы)')
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## Заморозка профиля: финальный выбор и его последствия

Среди SLO-eligible конфигураций на фронте выберите **самую быструю**.
Запишите прогноз held-out: что вы ожидаете увидеть на новых текстах?
Заморозьте профиль в файл с timestamp.

**Почему это финально:** после заморозки вы НЕ меняете профиль до конца ноутбука.
Если held-out покажет, что выбор неудачен — это честный результат (reject),
а не повод вернуться и выбрать другую конфигурацию. Изменение выбора после
увиденного held-out — это подгонка (tuning), не валидация (validation).

**Три честных вердикта:**
- **Accept** — профиль прошёл SLO на новых текстах. Ваш прогноз сработал (или
  был близок) — вы понимаете систему.
- **Reject** — не прошёл. Продукт требует меньшего обещания или большего железа.
  Сказать так с доказательствами — это работа инженера, а не поражение.
- **Inconclusive** — нельзя отличить «здоров» от «повезло». Профессиональный
  ответ — ещё один прогон, а не надежда в memo.

**Мемо-контракт для трёх сторон:**
- Продакт-владелец читает: «вот что мы обещаем при этой нагрузке»
- Capacity-планировщик читает: «вот где кривая гнётся и почему»
- Скептик читает: «вот альтернативное объяснение, которое этот прогон
  не исключает»

Все три вердикта — полноценные результаты. Оценивается заморозка, не удача.

In [ ]:
# Заморозка производственного профиля под SLO: p95 ≤ 800 мс, ошибки ≤ 5%.
_slo = dict(ttft_p95_ms=800.0, max_error_rate=0.01)
_eligible = [r for r in _front if r['p95_64'] <= _slo['ttft_p95_ms'] and r['err_64'] <= _slo['max_error_rate']]
chosen = max(_eligible, key=lambda r: r['thr_64']) if _eligible else None
frozen_name = chosen['name'] if chosen else 'NONE'
frozen_forecast = 'TODO: прогноз — что покажет held-out для этого профиля'
with open(os.path.join(RESULTS, 'profile.json'), 'w') as fh:
    json.dump(dict(name=frozen_name, forecast=frozen_forecast,
                   slo=_slo, frozen_at=time.time()), fh, ensure_ascii=False, indent=1)
print(f'профиль {frozen_name} заморожен — политика не меняется после этой точки')

In [ ]:
# Held-out: ДРУГИЕ тексты (V-сплит), тот же замороженный профиль — один прогон.
if chosen is None:
    print('ни один профиль не прошёл SLO — вердикт REJECT: продукт требует '
           'меньшего обещания или большей машины; held-out не выполняется')
else:
    heldout_recs = run_wave(MockTuningServer(max_num_seqs=chosen['seqs'],
                                              gpu_mem_util=chosen['mem'],
                                              chunked=chosen['chunked']),
                             'A', MOCK_CASES, 64, per_client=3, timeout_s=12.0)
    held_stats = _wave_stats(heldout_recs, 64)
    print(f"held-out: thr={held_stats['throughput']}, p95={held_stats['p95_ms']}мс, "
          f"timeouts={held_stats['timeouts']}/{held_stats['n']}" + chr(10) +
          'вердикт — TODO: accept / reject / inconclusive')

## Stretch: краевой риск и план отката

Каждый выбранный профиль имеет границу, за которой он перестаёт быть безопасным.
Здесь вы **намеренно** проверяете край: длиннейший промпт в корпусе (817 токенов)
против наименьшего KV-пула (mem=0.70). Обслуживается запрос или отклоняется?

**Краевой тест отвечает на вопрос:** «Что сломается первым при росте нагрузки
или длины промптов?» — до того, как это случится в продакшене.

**План отката** — это операционный артефакт, не формальность:
- **Trigger** — какая метрика, какой порог, как долго (не «станет медленно»,
  а «p95 > 800мс в течение 5 минут»)
- **Action** — что делать (revert to default / halve load / page on-call)
- **Evidence** — какой дашборд проверяет дежурный в 3 ночи (не «посмотрим логи»)

План без триггера — надежда. План без действия — наблюдение. План без evidence —
неисполним. Все три — это план.

In [ ]:
# Stretch: краевой тест — длиннейший промпт против наименьшего KV-пула.
_edge_case = dict(case_id='edge-long-prompt', input_tokens=817)
_edge_server = MockTuningServer(max_num_seqs=32, gpu_mem_util=0.70, chunked=True)
try:
    _edge_t, _edge_n = _edge_server.request(_edge_case, 32)
    print(f"краевой тест: длинный промпт ОБСЛУЖЕН ({_edge_n} токенов) при mem=0.70")
except Exception as e:
    print(f"краевой тест: длинный промпт ОТКЛОНЁН при mem=0.70: {type(e).__name__}")

In [ ]:
# Stretch: план отката — условие возврата к default.
rollback_trigger = 'TODO: метрика + порог + длительность (например: p95 > 800ms в течение 5 минут)'
rollback_action = 'TODO: действие при триггере (revert to default / halve load / page on-call)'
rollback_evidence = 'TODO: какой дашборд/запрос/порог проверяет дежурный в 3 ночи'
with open(os.path.join(RESULTS, 'rollback-plan.json'), 'w') as fh:
    json.dump(dict(trigger=rollback_trigger, action=rollback_action,
                   evidence=rollback_evidence), fh, ensure_ascii=False, indent=1)
print('план отката записан — заполните три TODO выше')

## Финальный эксперимент: ваш сервер против default

Судья вашего раунда оценивает **реальный** сервис: throughput на волнах 32 и 64
(корнем из произведения), p95 TTFT против SLO и долю ошибок. Всё, что было выше, —
поиск на мок-модели. Теперь та же дисциплина на живом сервисе: две конфигурации —
`default` и ваш замороженный профиль — по две волны судьи (32 и 64), один драйвер,
один журнал. На GPU-профиле сервер уже поднят с default-аргументами; замороженный
профиль применяется перезапуском сервера с ручками (vLLM принимает `--max-num-seqs`
только при старте). На mock-профиле тот же конвейер идёт через MockTuningServer —
числа останутся синтетикой, но ход анализа тот же.

**Вопрос на защиту:** прирост judge-метрики — это перевод ручки в механизм или
заслуга измерения? Что должен показать повторный прогон, прежде чем вы заявите
прирост в раунде (вспомните флип колена 64 → 32 → 64 из Темы 6)?

In [ ]:
# Финальный эксперимент: default против замороженного профиля, волны судьи 32/64.
# Оценка времени: mock ~30 с; GPU ~6–12 мин (один перезапуск сервера + 4 волны).
import math  # noqa: E402

def _gpu_apply_config(cfg: dict | None) -> None:
    """GPU ONLY: перезапустить owned-сервис с профилем (None = default-аргументы).

    Parameters
    ----------
    cfg : dict or None
        Строка профиля с seqs/mem/chunked; None — базовая команда без ручек.

    Returns
    -------
    None
        Перепривязывает глобальный server и ждёт /health (потолок 900 с).
    """
    global server
    server.terminate()
    try:
        server.wait(timeout=30)
    except subprocess.TimeoutExpired:
        server.kill()
        server.wait()
    cmd = list(SERVER_CMD)
    if cfg is not None:
        cmd += ['--max-num-seqs', str(cfg['seqs']),
                '--gpu-memory-utilization', str(cfg['mem'])]
        if cfg['chunked']:
            cmd.append('--enable-chunked-prefill')
    server = subprocess.Popen(cmd,
                              stdout=open(os.path.join(RESULTS, 'server.log'), 'a'),
                              stderr=subprocess.STDOUT)
    deadline = time.monotonic() + 900
    with tqdm(total=900, desc=f"health: {cfg['name'] if cfg else 'default'}", unit=' с') as bar:
        while True:
            try:
                with urllib.request.urlopen('http://127.0.0.1:8017/health', timeout=5) as r:
                    if r.status == 200:
                        break
            except OSError:
                pass
            assert time.monotonic() < deadline, 'service did not become healthy in 900 s'
            time.sleep(5)
            bar.update(5)

def judge_geomean(row: dict) -> float:
    """Оценка судьи раунда: корень из произведения throughput на волнах 32 и 64.

    Parameters
    ----------
    row : dict
        Строка финала с thr_32 и thr_64, tok/s.

    Returns
    -------
    float
        Геометрическое среднее throughput, tok/s.
    """
    # Уровень C: клетки выше уже прогнали обе конфигурации — объясните соседу,
    # почему судья берёт именно корень из произведения, а не среднее арифметическое.
    # Уровень B: реализуйте judge-метрику (math уже импортирован).
    # Уровень A: добавьте в _final_cfgs третьей строкой второй SLO-eligible
    # профиль из _front и одной фразой объясните, что меняет его присутствие.
    raise NotImplementedError('TODO: корень из произведения thr_32 * thr_64')

if chosen is None:
    print('ни один профиль не прошёл SLO — финальный эксперимент пропущен (вердикт REJECT)')
else:
    _final_rows = []
    _final_cfgs = [dict(name='default', seqs=256, mem=0.9, chunked=False), chosen]
    for cfg in tqdm(_final_cfgs, desc='финал', unit=' конфиг'):
        if PROFILE == 'gpu':
            _gpu_apply_config(None if cfg['name'] == 'default' else cfg)
            runner, cases = HttpRunner(), gpu_cases_A
        else:
            runner = MockTuningServer(max_num_seqs=cfg['seqs'],
                                      gpu_mem_util=cfg['mem'], chunked=cfg['chunked'])
            cases = MOCK_CASES
        recs = []
        for w in tqdm((32, 64), desc=f"  {cfg['name']}", unit=' волна', leave=False):
            recs.extend(run_wave(runner, 'A', cases, w, per_client=3, timeout_s=20.0))
        stats = {w: _wave_stats(recs, w) for w in (32, 64)}
        _final_rows.append(dict(
            name=cfg['name'], seqs=cfg['seqs'], mem=cfg['mem'], chunked=cfg['chunked'],
            thr_32=stats[32]['throughput'], thr_64=stats[64]['throughput'],
            p95_64=stats[64]['p95_ms'] if stats[64]['p95_ms'] is not None else 99999,
            err_64=stats[64]['timeouts'] / max(1, stats[64]['n'])))
    for row in _final_rows:
        row['judge_tok_s'] = judge_geomean(row)
    _gain = _final_rows[1]['judge_tok_s'] / _final_rows[0]['judge_tok_s']
    _slo_ok = _final_rows[1]['p95_64'] <= 800 and _final_rows[1]['err_64'] <= 0.01
    with open(os.path.join(RESULTS, 'final-experiment.json'), 'w') as fh:
        json.dump(dict(rows=_final_rows, gain=round(_gain, 3), slo_ok=_slo_ok,
                       profile_run=PROFILE), fh, ensure_ascii=False, indent=1)
    display(pd.DataFrame(_final_rows).style.set_caption(
        f'{"МОК" if PROFILE == "mock" else "GPU"}: прирост judge-метрики '
        f'x{_gain:.2f}; SLO-вердикт замороженного профиля: '
        f'{"PASS" if _slo_ok else "FAIL"} (final-experiment.json)'))

## Что вы построили — и чего этот прогон НЕ доказывает

**Артефакт:** замороженный производственный профиль с прогнозом, подтверждённый (или
опровергнутый) на held-out текстах, план отката с триггером, действием и каналом
проверки, и финальный эксперимент default-против-профиля с judge-метрикой —
всё в results/ с sha256-хэшами.

**Чему научились:** переводить каждую ручку vLLM в механизм из Topic 06;
объявлять пространство поиска ДО измерений; читать Парето-фронт «скорость против
хвоста»; замораживать выбор с фальсифицируемым прогнозом; валидировать на
held-out с тремя честными вердиктами; писать план отката, который дежурный
может исполнить в 3 ночи; измерять выигрыш профиля против default на живом
сервисе и связывать его с judge-метрикой раунда.

**Чего НЕ доказывает этот прогон:** mock — объявленная синтетика (её формула
написана в коде, а не открыта измерением); один прогон — не статистика;
burst-волны по 3 запроса на клиента — не устоявшаяся нагрузка; learner
validation не проводилась. GPU-профиль — единственный источник measured-чисел.

**Куда дальше:** Topic 08 — производственный API поверх замороженного профиля:
cancellation, quotas, backpressure. Профиль, который вы научились строить здесь, —
артефакт, который вы научитесь защищать там.

In [ ]:
artifacts = {}
for name in ('knob-table.json', 'search-plan.json', 'profile.json', 'raw-journal.jsonl',
             'final-experiment.json'):
    p = os.path.join(RESULTS, name)
    if os.path.exists(p):
        artifacts[name] = hashlib.sha256(Path_bytes(p)).hexdigest()
with open(os.path.join(RESULTS, 'manifest.json'), 'w') as fh:
    json.dump(dict(profile=PROFILE, artifacts=artifacts), fh, ensure_ascii=False, indent=1)
import shutil  # noqa: E402
shutil.make_archive('/content/t07-' + PROFILE + '-bundle', 'gztar', RESULTS)
print(f'{len(artifacts)} артефактов хэшировано; profile={PROFILE}; '
      f'bundle: /content/t07-{PROFILE}-bundle.tar.gz')

In [ ]:
# GPU PROFILE ONLY. Owned teardown of the service started in t07-gpu-bootstrap.
if PROFILE != 'gpu':
    print('skipped: gpu-only cell (profile is mock)')
else:
    server.terminate()
    try:
        server.wait(timeout=30)
    except subprocess.TimeoutExpired:
        server.kill()
        server.wait()
    print('server exit code', server.returncode,
          '- delete the runtime only after the bundle is downloaded')